# Computing Volume-Averaged and Integrated Flux

This tutorial uses `VolumePostprocessor` to reduce a scalar-flux field over the whole mesh and over a selected logical volume.

**Audience:** Users who have solved a transport problem and need quantitative results.

**Prerequisites:** A basic fixed-source calculation and logical volumes.

## Create a uniform reference solution

A uniform unit source in a purely absorbing material with reflecting boundaries produces a spatially uniform scalar flux of one. This makes the expected averages and integrals transparent.

In [ ]:
from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

rank = MPI.COMM_WORLD.rank
length = 4.0
nodes = [length * i / 40.0 for i in range(41)]
mesh = OrthogonalMeshGenerator(node_sets=[nodes]).Execute()
mesh.SetUniformBlockID(0)

xs = MultiGroupXS()
xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.0)
source = VolumetricSource(block_ids=[0], group_strength=[1.0])
quadrature = GLProductQuadrature1DSlab(n_polar=16, scattering_order=0)
problem = DiscreteOrdinatesProblem(
    mesh=mesh,
    num_groups=1,
    groupsets=[{"groups_from_to": (0, 0), "angular_quadrature": quadrature}],
    xs_map=[{"block_ids": [0], "xs": xs}],
    volumetric_sources=[source],
    boundary_conditions=[
        {"name": "zmin", "type": "reflecting"},
        {"name": "zmax", "type": "reflecting"},
    ],
)
solver = SteadyStateSourceSolver(problem=problem)
solver.Initialize()
solver.Execute()

## Reduce the solution

The whole-domain average should be one and its integral should equal the slab length. Restricting the integral to $1.0 \le z \le 2.5$ should give 1.5.

In [ ]:
whole_average = VolumePostprocessor(problem=problem, value_type="avg")
whole_integral = VolumePostprocessor(problem=problem, value_type="integral")
sample_region = RPPLogicalVolume(infx=True, infy=True, zmin=1.0, zmax=2.5)
regional_integral = VolumePostprocessor(
    problem=problem, value_type="integral", logical_volumes=[sample_region]
)
for postprocessor in (whole_average, whole_integral, regional_integral):
    postprocessor.Execute()

average = float(whole_average.GetValue()[0][0])
integral = float(whole_integral.GetValue()[0][0])
regional = float(regional_integral.GetValue()[0][0])
max_error = max(abs(average - 1.0), abs(integral - 4.0), abs(regional - 1.5))
if rank == 0:
    print(f"Whole-domain average={average:.6e}")
    print(f"Whole-domain integral={integral:.6e}")
    print(f"Regional integral={regional:.6e}")
    print(f"Volume-postprocessor max error={max_error:.6e}")
assert max_error < 1.0e-6
if "opensn_console" not in globals():
    from IPython import get_ipython
    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()